
# Lithospheric flexure from a real topographic profile

This notebook applies the 1-D elastic plate flexure equation to a real mountain-range cross section.

We will:

1. Fetch a topographic profile between two latitude/longitude coordinates.
2. Treat the topography as a surface load.
3. Fourier-transform the load.
4. Apply the elastic-plate response in wavenumber space.
5. Inverse-transform the result to obtain lithospheric deflection.

For a plate of constant flexural rigidity $D$,

$$
D\frac{d^4 w}{dx^4} + \Delta\rho g\,w = q(x),
$$

where

$$
D = \frac{E T_e^3}{12(1-\nu^2)}.
$$

After a Fourier transform,

$$
\hat w(k) =
\frac{\hat q(k)}
{Dk^4+\Delta\rho g}.
$$

The important idea is that an arbitrary load is just a sum of sinusoidal Fourier components. Each wavelength is flexed by a different amount.

**Sign convention used here:** $w>0$ means downward deflection.

### Elevation data

The profile is queried from the free [OpenTopoData](https://www.opentopodata.org/) public API using its global ETOPO1 dataset. This is deliberately low resolution because we are interested in mountain-range-scale wavelengths, not individual peaks.

The public service is rate-limited, so avoid repeatedly re-running the download cell unnecessarily.


In [ ]:

import numpy as np
import matplotlib.pyplot as plt
import requests

plt.rcParams["figure.figsize"] = (10, 4)

def pad_for_fft(values, factor=4):
    """
    Center a 1-D signal in a larger zero-padded array.

    Padding reduces interaction with periodic copies implied by the FFT.
    Returns the padded array and the slice that recovers the original data.
    """
    n = len(values)
    padded = np.zeros(factor * n, dtype=float)

    start = (len(padded) - n) // 2
    original_slice = slice(start, start + n)
    padded[original_slice] = values

    return padded, original_slice



## 1. Fetch a topographic cross section

Enter the two endpoints as `(latitude, longitude)`.

The function asks OpenTopoData for equally spaced samples along the line and then calculates cumulative distance using the haversine formula.

The default example crosses the central Andes near $20^\circ$S.


In [ ]:

def fetch_topographic_profile(point1, point2, n=100, dataset="etopo1"):
    """
    Fetch a topographic profile between two (lat, lon) points.

    Parameters
    ----------
    point1, point2 : tuple
        (latitude, longitude) in degrees.
    n : int
        Number of profile samples. Keep n <= 100 for the public API.
    dataset : str
        OpenTopoData dataset name.

    Returns
    -------
    distance_km, elevation_m, lat, lon : 1-D numpy arrays
    """
    if n > 100:
        raise ValueError("The public OpenTopoData API allows at most 100 sampled locations per request.")

    lat1, lon1 = point1
    lat2, lon2 = point2

    url = f"https://api.opentopodata.org/v1/{dataset}"
    params = {
        "locations": f"{lat1},{lon1}|{lat2},{lon2}",
        "samples": n,
        "interpolation": "bilinear",
    }

    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()
    data = r.json()

    if data.get("status") != "OK":
        raise RuntimeError(data.get("error", "Elevation request failed."))

    results = data["results"]
    elevation = np.array([p["elevation"] for p in results], dtype=float)
    lat = np.array([p["location"]["lat"] for p in results], dtype=float)
    lon = np.array([p["location"]["lng"] for p in results], dtype=float)

    if np.isnan(elevation).any():
        raise ValueError("The selected profile contains missing elevation values.")

    # Great-circle distance between successive samples
    R = 6371.0  # Earth radius [km]
    lat_r = np.radians(lat)
    lon_r = np.radians(lon)

    dlat = np.diff(lat_r)
    dlon = np.diff(lon_r)

    a = (
        np.sin(dlat / 2)**2
        + np.cos(lat_r[:-1]) * np.cos(lat_r[1:]) * np.sin(dlon / 2)**2
    )
    segment_km = 2 * R * np.arcsin(np.sqrt(a))
    distance_km = np.r_[0.0, np.cumsum(segment_km)]

    return distance_km, elevation, lat, lon


In [ ]:

# --- Choose your cross section here ---

point1 = (-20.0, -71.0)   # (latitude, longitude)
point2 = (-20.0, -61.0)

distance_km, elevation_m, lat, lon = fetch_topographic_profile(
    point1, point2, n=100
)

plt.plot(distance_km, elevation_m)
plt.axhline(0, linewidth=0.8)
plt.xlabel("Distance along profile [km]")
plt.ylabel("Elevation [m]")
plt.title("Topographic cross section")
plt.show()



## 2. Convert topography into a load

For this simple exercise, treat topography above a chosen reference level as rock added to the surface:

$$
q(x) = \rho_{\rm load} g h(x).
$$

This is deliberately simplified. A real mountain belt may contain crustal roots, density variations, erosion, sediment, water, pre-existing deformation, and spatially variable elastic thickness.

Choose endpoints in relatively low terrain so that the mountain range is reasonably isolated within the profile.

We also resample onto a perfectly uniform spatial grid, because the FFT assumes constant spacing.


In [ ]:

# Physical parameters
g = 9.81                 # m s^-2
rho_load = 2700.0        # kg m^-3, density of topographic load
rho_m = 3300.0           # kg m^-3
rho_c = 2800.0           # kg m^-3
delta_rho = rho_m - rho_c

E = 70e9                 # Pa
nu = 0.25
Te = 40e3                # elastic thickness [m]

D = E * Te**3 / (12 * (1 - nu**2))

# Uniform grid for the Fourier transform
x_km = np.linspace(distance_km[0], distance_km[-1], len(distance_km))
h = np.interp(x_km, distance_km, elevation_m)

# Only topography above the reference level is treated as load
reference_elevation = 0.0   # m
h_load = np.maximum(h - reference_elevation, 0.0)

q = rho_load * g * h_load   # Pa = N m^-2

print(f"Profile length: {x_km[-1]:.0f} km")
print(f"Elastic thickness Te: {Te/1000:.0f} km")
print(f"Flexural rigidity D: {D:.3e} N m")



## 3. Fourier transform the load

For a discretely sampled profile we use the real Fourier transform:

$$
q(x) \longrightarrow \hat{q}(k)
$$

The spatial wavenumber is

$$
k = \frac{2\pi}{\lambda}.
$$

Before the transform, the helper function defined at the top adds zero padding
to reduce artificial interaction between periodic copies of the profile.


In [ ]:
# Spatial spacing [m]
dx = (x_km[1] - x_km[0]) * 1000.0

# Prepare the load for the FFT
q_pad, profile_slice = pad_for_fft(q)

# Fourier transform
q_hat = np.fft.rfft(q_pad)
k = 2 * np.pi * np.fft.rfftfreq(len(q_pad), d=dx)

# Convert wavenumber to wavelength for plotting
wavelength_km = np.full_like(k, np.nan, dtype=float)
wavelength_km[1:] = (2 * np.pi / k[1:]) / 1000.0

plt.loglog(wavelength_km[1:], np.abs(q_hat[1:]))
plt.gca().invert_xaxis()
plt.xlabel("Wavelength [km]")
plt.ylabel(r"$|\hat{q}|$")
plt.title("Fourier spectrum of the topographic load")
plt.show()


## 4. Apply the flexural response in Fourier space

For each wavenumber,

$$
\hat w(k)
=
\frac{\hat q(k)}
{Dk^4+\Delta\rho g}.
$$

The transfer function

$$
H(k)=\frac{1}{Dk^4+\Delta\rho g}
$$

shows why long wavelengths flex the plate much more efficiently than short wavelengths.


In [ ]:

# Flexural transfer function
H = 1.0 / (D * k**4 + delta_rho * g)

# Deflection in Fourier space
w_hat = H * q_hat

# Plot the response relative to local isostatic compensation
compensation = (delta_rho * g) * H

plt.semilogx(wavelength_km[1:], compensation[1:])
plt.xlabel("Wavelength [km]")
plt.ylabel("Relative compensation")
plt.ylim(0, 1.05)
plt.title("Flexural response as a function of wavelength")
plt.grid(alpha=0.3)
plt.show()



## 5. Inverse Fourier transform

Now transform $\hat w(k)$ back into physical space:

$$
\hat w(k) \longrightarrow w(x).
$$

The result is the downward deflection predicted by the elastic-plate model.


In [ ]:
# Inverse Fourier transform
w_pad = np.fft.irfft(w_hat, n=len(q_pad))

# Keep only the part corresponding to our topographic profile
w_m = w_pad[profile_slice]

fig, ax = plt.subplots()

ax.plot(x_km, h / 1000, label="Topography")
ax.plot(x_km, -w_m / 1000, label="Flexural deflection")
ax.axhline(0, linewidth=0.8)

ax.set_xlabel("Distance along profile [km]")
ax.set_ylabel("Elevation / deflection [km]")
ax.set_title(f"Elastic plate flexure, $T_e$ = {Te/1000:.0f} km")
ax.legend()
plt.show()


## 6. Things to try

Change **one thing at a time** and predict the result before running the cell again.

- Change $T_e$. What happens when the plate is 10, 30, 50, or 80 km thick?
- Choose a wider or narrower mountain range.
- Compare two mountain belts.
- Look at the Fourier spectrum: which wavelengths dominate the load?
- Compare the flexural response curve with those wavelengths.
- Explain why changing $T_e$ has such a strong effect using
  $$
  D \propto T_e^3.
  $$

### Questions

1. Why are short-wavelength loads weakly compensated?
2. Why does the deflection extend beyond the region where the surface load is non-zero?
3. What does the $k=0$ component represent?
4. What assumptions in this model are least realistic for an actual mountain range?
5. What would have to change to solve this problem in 2-D?



## Optional: compare several elastic thicknesses

This is useful after you understand the single-$T_e$ calculation above.


In [ ]:
Te_values_km = [10, 30, 50, 80]

plt.plot(x_km, h / 1000, linewidth=2, label="Topography")

for Te_km in Te_values_km:
    Te_i = Te_km * 1000.0
    D_i = E * Te_i**3 / (12 * (1 - nu**2))
    H_i = 1.0 / (D_i * k**4 + delta_rho * g)

    w_i_pad = np.fft.irfft(q_hat * H_i, n=len(q_pad))
    w_i = w_i_pad[profile_slice]

    plt.plot(x_km, -w_i / 1000, label=f"Te = {Te_km} km")

plt.axhline(0, linewidth=0.8)
plt.xlabel("Distance along profile [km]")
plt.ylabel("Elevation / deflection [km]")
plt.title("Effect of elastic thickness")
plt.legend()
plt.show()


---

### Notes on interpretation

This notebook is intended to demonstrate the **spectral mechanics of plate flexure**, not to reproduce the full tectonic structure of a mountain belt.

The main assumptions are:

- thin elastic plate,
- constant and isotropic $D$,
- small deflections/slopes,
- constant density contrast,
- topography treated as an applied surface load,
- 1-D geometry,
- no horizontal plate force $P$,
- no time dependence or viscoelastic relaxation.

The FFT still implies periodic boundary conditions; the helper function uses padding only to reduce edge interaction.
